In [64]:
!pip install ultralytics

In [65]:
from collections import defaultdict

import cv2
import numpy as np
from google.colab.patches import cv2_imshow
from ultralytics import YOLO
from google.colab import drive
drive.mount('/content/drive')

# Load the YOLOv8 model
model = YOLO('yolov8n.pt')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [67]:
import csv
import pandas as pd
import random
import datetime
import os

folder_path ="/content/drive/MyDrive/0_497Project/preliminarydata/splittingzoom9"

counter_frames=0
df = pd.DataFrame(columns=["Video name", "Other pedestrians","Vehicle Around" ])
for file_name in os.listdir(folder_path):
        if file_name.endswith(".avi"):  # Check if the file is a .mov video
            video_path = os.path.join(folder_path, file_name)
            print(f"Processing video: {video_path}")
            # Open the video
            cap = cv2.VideoCapture(video_path)

            # Store the track history
            #defaultdict that stores the history of tracked objects
            track_history = defaultdict(lambda: [])
            type_dict = {'person':[], 'car':[]}
            # Loop through the video frames
            #It's a loop that processes each frame of the video until it reaches the end or the user presses
            while cap.isOpened():
                # Read a frame from the video
                success, frame = cap.read()


                if success:
                    # Run YOLOv8 tracking on the frame, persisting tracks between frames
                    #It retrieves the bounding boxes and track IDs of detected objects.
                    results = model.track(frame, persist=True)
                    if results[0].boxes.id != None:
                      #print(results[0].boxes)
                      # Get the boxes and track IDs
                      boxes = results[0].boxes.xywh.cpu()
                      track_ids = results[0].boxes.id.int().cpu().tolist()

                      # Visualize the results on the frame
                      annotated_frame = results[0].plot()
                      #save tracking history
                      if len(track_ids)>1:
                        for i in range(len(track_ids)):
                          type_id = int(results[0].boxes[i].cls[0])
                          if type_id==0:
                            if i in type_dict['person']:
                              pass
                            else:
                              type_dict['person'].append(i)
                          elif type_id==2:
                            if i in type_dict['car']:
                              pass
                            else:
                              type_dict['car'].append(i)



                    else:
                      continue

                    # Plot the tracks
                    for box, track_id in zip(boxes, track_ids):
                        x, y, w, h = box
                        track = track_history[track_id]
                        track.append((float(x), float(y)))  # x, y center point
                        if len(track) > 30:  # retain 90 tracks for 90 frames
                            track.pop(0)

                        # Draw the tracking lines
                        points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
                        cv2.polylines(annotated_frame, [points], isClosed=False, color=(230, 230, 230), thickness=10)

                    # Display the annotated frame
                    #cv2_imshow( annotated_frame)#"YOLOv8 Tracking",

                    # Break the loop if 'q' is pressed
                    if cv2.waitKey(1) & 0xFF == ord("q"):
                        break
                else:
                    # Break the loop if the end of the video is reached
                    break
                counter_frames += 1
            # Release the video capture object and close the display window
            cap.release()
            cv2.destroyAllWindows()



            df = df.append({"Video name": file_name, "Other pedestrians": len(type_dict['person'])-1, "Vehicle Around": len(type_dict['car'])}, ignore_index=True)


Processing video: /content/drive/MyDrive/0_497Project/preliminarydata/splittingzoom9/output_video_1.avi

0: 384x640 1 person, 4 cars, 238.9ms
Speed: 3.6ms preprocess, 238.9ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 4 cars, 179.7ms
Speed: 3.0ms preprocess, 179.7ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 4 cars, 169.2ms
Speed: 5.5ms preprocess, 169.2ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 4 cars, 167.8ms
Speed: 6.4ms preprocess, 167.8ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 4 cars, 179.7ms
Speed: 4.7ms preprocess, 179.7ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 4 cars, 157.0ms
Speed: 4.3ms preprocess, 157.0ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 4 cars, 175.0ms
Speed: 4.8ms preprocess, 175.0ms inference

<ipython-input-67-db9806cee80f>:89: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append({"Video name": file_name, "Other pedestrians": len(type_dict['person'])-1, "Vehicle Around": len(type_dict['car'])}, ignore_index=True)



0: 384x640 2 cars, 1 stop sign, 161.7ms
Speed: 2.7ms preprocess, 161.7ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 1 stop sign, 162.3ms
Speed: 7.6ms preprocess, 162.3ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 boat, 1 stop sign, 163.2ms
Speed: 6.1ms preprocess, 163.2ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 1 stop sign, 151.3ms
Speed: 6.1ms preprocess, 151.3ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 1 stop sign, 157.4ms
Speed: 6.0ms preprocess, 157.4ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 1 stop sign, 153.2ms
Speed: 5.8ms preprocess, 153.2ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 1 stop sign, 158.8ms
Speed: 5.1ms preprocess, 158.8ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars,

<ipython-input-67-db9806cee80f>:89: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append({"Video name": file_name, "Other pedestrians": len(type_dict['person'])-1, "Vehicle Around": len(type_dict['car'])}, ignore_index=True)



0: 384x640 2 cars, 1 stop sign, 257.8ms
Speed: 4.0ms preprocess, 257.8ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 stop sign, 236.9ms
Speed: 5.3ms preprocess, 236.9ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 stop sign, 156.0ms
Speed: 10.8ms preprocess, 156.0ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 stop sign, 157.5ms
Speed: 5.3ms preprocess, 157.5ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 stop sign, 175.9ms
Speed: 5.1ms preprocess, 175.9ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 stop sign, 159.7ms
Speed: 6.1ms preprocess, 159.7ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 stop sign, 155.8ms
Speed: 5.0ms preprocess, 155.8ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 stop

...

<ipython-input-67-db9806cee80f>:89: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append({"Video name": file_name, "Other pedestrians": len(type_dict['person'])-1, "Vehicle Around": len(type_dict['car'])}, ignore_index=True)



0: 384x640 8 persons, 1 car, 1 bus, 1 truck, 1 stop sign, 169.9ms
Speed: 4.5ms preprocess, 169.9ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 truck, 173.5ms
Speed: 3.2ms preprocess, 173.5ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 truck, 170.4ms
Speed: 3.3ms preprocess, 170.4ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 truck, 189.1ms
Speed: 5.2ms preprocess, 189.1ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 truck, 168.1ms
Speed: 3.1ms preprocess, 168.1ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 truck, 177.8ms
Speed: 2.7ms preprocess, 177.8ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 truck, 169.9ms
Speed: 3.3ms preprocess, 169.9ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 truck, 178.2ms
Speed: 2.7ms preprocess, 178.2ms inference, 

<ipython-input-67-db9806cee80f>:89: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append({"Video name": file_name, "Other pedestrians": len(type_dict['person'])-1, "Vehicle Around": len(type_dict['car'])}, ignore_index=True)



0: 384x640 1 car, 168.2ms
Speed: 2.8ms preprocess, 168.2ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 1 car, 186.6ms
Speed: 4.5ms preprocess, 186.6ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 1 car, 181.6ms
Speed: 5.2ms preprocess, 181.6ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 1 car, 168.8ms
Speed: 6.3ms preprocess, 168.8ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 1 car, 169.6ms
Speed: 5.4ms preprocess, 169.6ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 1 car, 181.0ms
Speed: 6.9ms preprocess, 181.0ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 1 car, 185.1ms
Speed: 5.4ms preprocess, 185.1ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 1 car, 180.9ms
Speed: 5.9ms preprocess, 180

<ipython-input-67-db9806cee80f>:89: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append({"Video name": file_name, "Other pedestrians": len(type_dict['person'])-1, "Vehicle Around": len(type_dict['car'])}, ignore_index=True)


In [68]:

df.to_csv("/content/drive/MyDrive/0_497Project/preliminarydata/splittingzoom9/context_videos.csv", index=False)
